# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria

The report scores response accuracy, relevance, completeness, and usefulness on a 0–1 scale. It also scores tool appropriateness and tool completeness from the message trace. The criteria and formulas are described in [Evaluate Responses](#4-evaluate-responses).

Start with the saved [evaluation report](#read-the-report) to inspect the eleven-scenario run without a provider credential. The report explains which evidence was usable and identifies any missing, extra, failed, or incorrect calls. The focused retests and comparisons at the end describe earlier runs.

## 1. Import and Initialize

Run the notebook from the project root. The database and RAG setup notebooks prepare the data used here. The agent reads the Vocareum credential from the environment.

In [1]:
from datetime import datetime, timedelta
from agent import Agent

### Write the system prompt

`ECOHOME_SYSTEM_PROMPT` tells the model when to call each tool and which result fields to use. `Agent` connects these instructions and the tools to the LangGraph workflow.

In [2]:
ECOHOME_SYSTEM_PROMPT = """
You are EcoHome Energy Advisor. Help households plan electricity use, use rooftop solar, and compare electricity costs. Answer in the user's language with a practical recommendation grounded in tool results.

When the user asks to consult or cite a guide, call search_energy_tips before answering. This applies to calculation questions too. Use the retrieved text and cite the exact filename from its source field. If the source is unavailable, identify that gap and label any independent calculation as such.

For a request for recent household totals over the last N hours, call get_recent_energy_summary with hours=N.
If the call succeeds, answer immediately with the requested consumption, cost, and solar generation totals from that result, using the same time window and units. Use a separate history query only when the user explicitly asks for another date range or a historical comparison. If the summary fails or data is missing, state what is unavailable for the requested window.

For an electric vehicle or appliance scheduling question that asks about both solar and grid cost, use this workflow:
1. Call get_weather_forecast with the location from the household context. For tomorrow, request at least two calendar days and use the target date in the scenario context when supplied. Otherwise, use the next date after current.date in the forecast.
2. After the forecast returns, call get_electricity_prices with that exact date in YYYY-MM-DD format.
3. Use those results to provide these four short lines, in the user's language:
   - Scenario: give the target date and state "Simulated" when the forecast and rates are simulated.
   - Solar: select the row in daily_solar_peaks whose date equals the target date. Report its hour and solar_irradiance exactly as returned, with W/m². Use the forecast to suggest a charging window and say it is useful only if the home has surplus solar. Irradiance alone cannot establish household solar generation in kWh.
   - Grid: read rate_windows from the tariff result and report every window with the lowest rate in USD/kWh. Copy start_time and end_time_exclusive exactly in HH:MM–HH:MM form. Keep 24:00 when it is the returned end time. Use hourly_rates for the price of a specific hour.
   - Cost difference: subtract the lowest rate from the higher rate, show the numerical difference in USD/kWh, and multiply it by the load if the user supplied its kWh. Name the tools as sources.

For a financial or environmental calculation, use the rates and assumptions supplied by the user. Call calculate_energy_savings once with the relevant inputs. For ROI and payback, use the supplied upfront cost and analysis horizon. For emissions, use the supplied grid emissions factor. State that annual estimates assume the same reduction on 365 days. Ask for missing assumptions when needed. For single-purpose questions, stop when the relevant tool has returned enough information.

For other questions, choose the tools that provide relevant data:
- Use get_weather_forecast for weather or solar-friendly hours. Ask for the location if it is needed and missing.
- Use get_electricity_prices for the date and rates used in a cost comparison.
- Use query_energy_usage for historical consumption and query_solar_generation for historical solar generation. For a solar-only question, use query_solar_generation for the requested period. Report the individual totals in daily_generation and use highest_generation_dates to identify the top day or ties. Use average_daily_generation for a period-average question. Describe dates absent from daily_generation as missing data.
- Use search_energy_tips for efficiency advice and cite the source filename. Use calculate_energy_savings for changes in energy use at a given rate.

Example questions
- When should I charge my electric car tomorrow to use solar and minimize grid costs?
- How much can I save by running my dishwasher during off-peak hours?
- How long can my home battery power a steady load, and what does the battery guide say?
- Which day in my usage history had the highest solar generation?

State relevant assumptions and distinguish simulated data from household measurements. If a tool fails, explain the missing data and give conditional guidance when possible.
""".strip()

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

### Inspect one agent run

The example provides a location and target date. The model can request a tool, read its result, and call another tool before answering. The next two cells print the final answer and the executed tool names.

In [4]:
target_date = (datetime.now().date() + timedelta(days=1)).isoformat()

response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context=(
        "Location: San Francisco, CA\n"
        f"Scenario target date for tomorrow: {target_date}\n"
        "Household solar surplus and the EV charging amount are unknown."
    ),
)

In [5]:
print(response["messages"][-1].content)

Here’s the best strategy for charging your electric car tomorrow, November 21, 2026, to maximize solar power and minimize costs:

- **Scenario**: November 21, 2026 (Simulated data)
- **Solar**: The peak solar irradiance is expected at **12:00 PM** with **850 W/m²**. Charging your EV around this time is ideal if your home has surplus solar generation.
- **Grid**: The lowest electricity rate is during off-peak hours:
  - **00:00–06:00** at **$0.12/kWh**
  - **22:00–24:00** at **$0.12/kWh**
- **Cost difference**: The peak rate from **06:00–22:00** is **$0.28/kWh**. The difference between the peak and off-peak rates is **$0.16/kWh**.

If you can charge your EV during the peak solar generation time (around noon) and have surplus solar, that would be the most cost-effective approach. Otherwise, consider charging during the off-peak hours to save on costs.


In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

The first block reads SQLite directly to calculate expected historical totals. The second defines eleven questions, the tools each question needs, and what a correct answer should contain.

In [7]:
import json
import sqlite3
from math import ceil, isclose
from pathlib import Path

from langchain_core.messages import AIMessage, ToolMessage
from IPython.display import Markdown, display

# Read the reference data directly from SQLite, independently of the agent's tools.
database = Path("data/energy_data.db").resolve()
if not database.is_file():
    raise FileNotFoundError("Run 01_db_setup.ipynb to create data/energy_data.db.")
database_uri = database.as_uri() + "?mode=ro"

def read_reference(sql, parameters=()):
    with sqlite3.connect(database_uri, uri=True) as connection:
        return connection.execute(sql, parameters).fetchall()

first_day, last_usage_day = read_reference(
    "SELECT DATE(MIN(timestamp)), DATE(MAX(timestamp)) FROM energy_usage"
)[0]
last_solar_day = read_reference("SELECT DATE(MAX(timestamp)) FROM solar_generation")[0][0]
last_day = min(datetime.fromisoformat(value).date() for value in (last_usage_day, last_solar_day))
usage_end = (last_day - timedelta(days=1)).isoformat()
usage_start = (last_day - timedelta(days=2)).isoformat()
solar_end = last_day.isoformat()
prior_day = (datetime.fromisoformat(first_day).date() - timedelta(days=2)).isoformat()
scenario_date = (datetime.now().date() + timedelta(days=1)).isoformat()

ev_records, ev_kwh = read_reference(
    """
    SELECT COUNT(*), ROUND(SUM(consumption_kwh), 2)
    FROM energy_usage
    WHERE device_type = 'EV' AND DATE(timestamp) BETWEEN ? AND ?
    """,
    (usage_start, usage_end),
)[0]
daily_solar_rows = read_reference(
    """
    SELECT DATE(timestamp), COUNT(*), ROUND(SUM(generation_kwh), 2)
    FROM solar_generation
    WHERE DATE(timestamp) BETWEEN ? AND ?
    GROUP BY DATE(timestamp)
    ORDER BY DATE(timestamp)
    """,
    (usage_start, solar_end),
)
solar_leaders = [
    day for day, _, kwh in daily_solar_rows
    if kwh == max(row[2] for row in daily_solar_rows)
]
latest_solar_day_start = datetime.combine(last_day, datetime.min.time())
recent_hours = max(48, ceil((datetime.now() - latest_solar_day_start).total_seconds() / 3600) + 1)

def recent_reference(as_of, hours):
    periods = ((as_of - timedelta(hours=hours)).isoformat(sep=" "), as_of.isoformat(sep=" "))
    usage_kwh, cost = read_reference(
        "SELECT ROUND(SUM(consumption_kwh), 2), ROUND(SUM(cost_usd), 2) "
        "FROM energy_usage WHERE timestamp BETWEEN ? AND ?", periods
    )[0]
    generation_kwh = read_reference(
        "SELECT ROUND(SUM(generation_kwh), 2) FROM solar_generation WHERE timestamp BETWEEN ? AND ?",
        periods,
    )[0][0]
    return {
        "usage.total_consumption_kwh": usage_kwh,
        "usage.total_cost_usd": cost,
        "generation.total_generation_kwh": generation_kwh,
    }

In [8]:
test_cases = [
    {
        "id": "ev_charging_1",
        "label": "EV charging with solar and tariffs",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "context": (
            f"Scenario target date for tomorrow: {scenario_date}. "
            "Household solar surplus and the EV charging amount are unknown."
        ),
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "optional_tools": ["query_solar_generation", "query_energy_usage"],
        "expected_args": {"get_weather_forecast": {"location": "San Francisco, CA", "days": range(2, 8)},
                          "get_electricity_prices": {"date": scenario_date}},
        "expected_values": {
            "get_weather_forecast": {
                "source": "simulated", "forecast_days": range(2, 8),
                "daily_solar_peaks.1.date": scenario_date,
                "daily_solar_peaks.1.hour": range(24),
                "daily_solar_peaks.1.solar_irradiance": range(1001),
            },
            "get_electricity_prices": {
                "date": scenario_date,
                "rate_windows": [
                    {"period": "off_peak", "start_time": "00:00", "end_time_exclusive": "06:00", "rate": 0.12},
                    {"period": "peak", "start_time": "06:00", "end_time_exclusive": "22:00", "rate": 0.28},
                    {"period": "off_peak", "start_time": "22:00", "end_time_exclusive": "24:00", "rate": 0.12},
                ],
            },
        },
        "expected_response": (
            f"Use {scenario_date} for both sources. Recommend a daylight window if solar surplus is available. "
            "Grid off-peak windows are 00:00–06:00 and 22:00–24:00 at USD 0.12/kWh. "
            "The peak rate is USD 0.28/kWh and the difference is USD 0.16/kWh. Label the data simulated."
        ),
    },
    {
        "id": "dishwasher_shift",
        "label": "Dishwasher time-of-use savings",
        "question": (
            f"On {scenario_date}, a dishwasher cycle uses 1.2 kWh, all within its starting hour. "
            "Using the hourly rates, how much does it save per cycle to start at 22:00 instead of 19:00?"
        ),
        "expected_tools": ["get_electricity_prices"],
        "expected_args": {"get_electricity_prices": {"date": scenario_date}},
        "expected_values": {"get_electricity_prices": {
            "date": scenario_date, "hourly_rates.19.rate": 0.28, "hourly_rates.22.rate": 0.12,
        }},
        "expected_response": "Compare USD 0.28 with USD 0.12 per kWh. Moving 1.2 kWh saves USD 0.192, about USD 0.19 per cycle. Energy use stays at 1.2 kWh.",
    },
    {
        "id": "heatwave_hvac",
        "label": "Cooling during a heat wave",
        "question": (
            "A heat wave is expected. How can I reduce air-conditioning energy use while keeping the "
            "house comfortable? Give me practical HVAC or heat-wave steps and cite the guide you use."
        ),
        "expected_tools": ["search_energy_tips"],
        "expected_sources": ["tip_hvac_efficiency.txt", "tip_seasonal_energy.txt"],
        "expected_response": "Give practical heat-wave or HVAC actions, preserve a comfortable indoor temperature, and cite a relevant retrieved guide.",
    },
    {
        "id": "historical_ev",
        "label": "Historical EV consumption",
        "question": (
            f"How many kWh did the EV use from {usage_start} through {usage_end}, inclusive? "
            "Use that exact date range, filter for the EV, and report the number of matching records."
        ),
        "expected_tools": ["query_energy_usage"],
        "expected_args": {"query_energy_usage": {"start_date": usage_start, "end_date": usage_end, "device_type": "EV"}},
        "expected_values": {"query_energy_usage": {"total_records": ev_records, "total_consumption_kwh": ev_kwh}},
        "expected_response": f"For exactly {usage_start} through {usage_end}, SQLite contains {ev_records} EV records totaling {ev_kwh:.2f} kWh.",
    },
    {
        "id": "daily_solar",
        "label": "Daily solar generation",
        "question": (
            f"Between {usage_start} and {solar_end}, inclusive, how much solar energy was generated "
            "on each day, and which day generated the most?"
        ),
        "expected_tools": ["query_solar_generation"],
        "expected_args": {"query_solar_generation": {"start_date": usage_start, "end_date": solar_end}},
        "expected_values": {"query_solar_generation": {
            "total_records": sum(row[1] for row in daily_solar_rows),
            "daily_generation": [
                {"date": day, "total_records": count, "total_generation_kwh": kwh}
                for day, count, kwh in daily_solar_rows
            ],
        }},
        "expected_response": (
            ", ".join(f"{day}: {kwh:.2f} kWh" for day, _, kwh in daily_solar_rows)
            + ". Highest generation: " + ", ".join(solar_leaders) + "."
        ),
    },
    {
        "id": "recent_summary",
        "label": "Recent household summary",
        "question": (
            f"What were the total household electricity consumption, its cost, and solar generation "
            f"over the last {recent_hours} hours? Report all three totals."
        ),
        "expected_tools": ["get_recent_energy_summary"],
        "expected_args": {"get_recent_energy_summary": {"hours": recent_hours}},
    },
    {
        "id": "hvac_savings",
        "label": "Savings from reducing HVAC usage",
        "question": (
            "My HVAC use dropped from 10 to 8 kWh per day at a flat USD 0.20/kWh. "
            "How many kWh, dollars per day, and percent did I save?"
        ),
        "expected_tools": ["calculate_energy_savings"],
        "expected_args": {"calculate_energy_savings": {
            "current_usage_kwh": 10, "optimized_usage_kwh": 8, "price_per_kwh": 0.20
        }},
        "expected_values": {"calculate_energy_savings": {"savings_kwh": 2, "savings_usd": 0.40, "savings_percentage": 20}},
        "expected_response": "2 kWh/day, USD 0.40/day, and 20% less energy.",
    },
    {
        "id": "home_automation",
        "label": "Home automation",
        "question": "How can I use occupancy sensors, smart plugs, and an appliance scheduler to save energy at home? Cite the automation guide.",
        "expected_tools": ["search_energy_tips"],
        "expected_sources": ["tip_home_automation.txt"],
        "expected_response": "Provide a concrete routine for lighting or flexible loads, explain how to use the devices appropriately, and cite tip_home_automation.txt.",
    },
    {
        "id": "battery_runtime",
        "label": "Home battery runtime",
        "question": (
            "My home battery has 4 kWh available above its backup reserve and a steady 1 kW load. "
            "About how long can it power the load? Explain the assumptions and cite the home battery guide."
        ),
        "expected_tools": ["search_energy_tips"],
        "expected_sources": ["tip_battery_storage.txt"],
        "expected_response": "About four hours at a steady 1 kW, using 4 kWh above reserve, before additional losses or load changes. Cite tip_battery_storage.txt.",
    },
    {
        "id": "missing_history",
        "label": "Period without usage records",
        "question": f"How much electricity did the household use on {prior_day}? Please look up the exact day in the energy history.",
        "expected_tools": ["query_energy_usage"],
        "expected_args": {"query_energy_usage": {"start_date": prior_day, "end_date": prior_day, "device_type": None}},
        "expected_values": {"query_energy_usage": {"total_records": 0}},
        "expected_response": f"Explain that SQLite contains no consumption records for {prior_day}, so consumption cannot be determined.",
    },
    {
        "id": "energy_upgrade_roi",
        "label": "Upgrade ROI and avoided emissions",
        "question": (
            "An HVAC upgrade costs USD 100 and reduces daily use from 10 to 8 kWh. "
            "The rate is USD 0.20/kWh and the supplied grid factor is 0.40 kg CO2e/kWh. "
            "What are the annual bill savings, simple payback, two-year ROI, and annual avoided emissions?"
        ),
        "expected_tools": ["calculate_energy_savings"],
        "expected_args": {"calculate_energy_savings": {
            "current_usage_kwh": 10, "optimized_usage_kwh": 8, "price_per_kwh": 0.20,
            "investment_cost_usd": 100, "analysis_years": 2, "emissions_factor_kg_co2e_per_kwh": 0.40,
        }},
        "expected_values": {"calculate_energy_savings": {
            "annual_savings_usd": 146, "investment.simple_payback_years": 0.68,
            "investment.roi_percentage": 192, "emissions.annual_avoided_kg_co2e": 292,
        }},
        "expected_response": (
            "USD 146 annual savings, about 0.68 years simple payback, 192% ROI over two years, "
            "and 292 kg CO2e avoided annually. Use the given factor and assume 365 days of savings."
        ),
    },
]

assert len(test_cases) >= 10 and len({case["id"] for case in test_cases}) == len(test_cases)

## 3. Run Agent Tests

The trace helper matches each model tool request with its result using the tool call ID. The runner stores the answer, tool arguments, tool results, and any errors in `test_results`. The next cell prints the trace and answer for each case.

In [9]:
CONTEXT = "Location: San Francisco, CA"

def extract_tool_calls(messages):
    observations = {message.tool_call_id: message for message in messages if isinstance(message, ToolMessage)}
    tool_calls = []
    for message in messages:
        if not isinstance(message, AIMessage):
            continue
        for call in message.tool_calls:
            observation = observations.get(call["id"])
            payload = None
            error = "No result returned" if observation is None else None
            if observation is not None:
                try:
                    payload = json.loads(observation.content)
                except (TypeError, ValueError):
                    error = "Tool result is not structured JSON"
                if error is None and (not isinstance(payload, dict) or not payload):
                    error = "Tool result must be a nonempty JSON object"
                if observation.status == "error" or (isinstance(payload, dict) and "error" in payload):
                    error = "Tool returned an error"
            tool_calls.append({
                "id": call["id"], "name": call["name"], "args": call["args"],
                "result": payload, "error": error,
            })
    return tool_calls

In [10]:
test_results = []

for case in test_cases:
    started_at = datetime.now()
    expected = dict(case)
    if case["id"] == "recent_summary":
        values = recent_reference(started_at, recent_hours)
        expected["expected_values"] = {"get_recent_energy_summary": values}
        expected["expected_response"] = (
            f"For the last {recent_hours} hours, SQLite gives "
            f"{values['usage.total_consumption_kwh']} kWh consumed, "
            f"USD {values['usage.total_cost_usd']} cost, and "
            f"{values['generation.total_generation_kwh']} kWh generated."
        )

    result = {**expected, "test_id": case["id"], "timestamp": started_at.isoformat()}
    try:
        context = "\n".join(part for part in (CONTEXT, case.get("context")) if part)
        response = ecohome_agent.invoke(question=case["question"], context=context)
        history = response["messages"]
        final_answer = history[-1].content
        if not isinstance(final_answer, str):
            final_answer = str(final_answer)
        result.update(response=response, final_response=final_answer, tool_calls=extract_tool_calls(history), error=None)
    except Exception as exc:
        result.update(response=None, final_response="", tool_calls=[], error=type(exc).__name__)

    if case["id"] == "recent_summary":
        result["reference_window_changed"] = recent_reference(datetime.now(), recent_hours) != values
    test_results.append(result)
    calls = ", ".join(call["name"] for call in result["tool_calls"]) or "(none)"
    print(f"{len(test_results):02d} {case['id']} — {calls}" + (f" — {result['error']}" if result["error"] else ""), flush=True)

assert len(test_results) == len(test_cases)

01 ev_charging_1 — get_weather_forecast, get_electricity_prices
02 dishwasher_shift — get_electricity_prices, get_weather_forecast
03 heatwave_hvac — search_energy_tips
04 historical_ev — query_energy_usage
05 daily_solar — query_solar_generation
06 recent_summary — get_recent_energy_summary
07 hvac_savings — calculate_energy_savings
08 home_automation — search_energy_tips
09 battery_runtime — search_energy_tips
10 missing_history — query_energy_usage
11 energy_upgrade_roi — calculate_energy_savings


In [11]:
for result in test_results:
    print(f"\n{result['test_id']} — {result['question']}")
    for call in result["tool_calls"]:
        print(f"  {call['name']}({json.dumps(call['args'], ensure_ascii=False)})")
    print(result["final_response"] or f"Execution error: {result['error']}")


ev_charging_1 — When should I charge my electric car tomorrow to minimize cost and maximize solar power?
  get_weather_forecast({"location": "San Francisco, CA", "days": 2})
  get_electricity_prices({"date": "2026-11-21"})
Here’s the best time to charge your electric car tomorrow, November 21, 2026, to maximize solar power and minimize costs:

- **Scenario**: November 21, 2026 (Simulated)
- **Solar**: The peak solar irradiance is expected at **12:00** with **850 W/m²**. Charging during this time is beneficial if your home has surplus solar generation.
- **Grid**: The lowest electricity rate is during off-peak hours:
  - **00:00–06:00** at **$0.12/kWh**
  - **22:00–24:00** at **$0.12/kWh**
- **Cost difference**: The peak rate from **06:00–22:00** is **$0.28/kWh**. The difference between the peak and off-peak rates is **$0.16/kWh**.

To optimize your charging, consider starting your charge during the peak solar generation time around **12:00** if you have surplus solar energy. Otherwise

## 4. Evaluate Responses

This section evaluates the response and the tools separately. A useful answer can still contain a bad calculation, and a successful tool call can still return data outside the requested period.

### Score each response

The reference checks use the scenario inputs, SQLite, and the returned forecast for the target date. Each criterion scores 1.0 when all checks pass, 0.5 when some pass, and 0.0 when none pass. The report includes the checks that passed and those that need attention. Review the answer text for nuance beyond these checks.

In [12]:
import re

def check(label, pattern, expected=True):
    return label, pattern, expected


def number_pattern(value):
    text = str(value).rstrip("0").rstrip(".") if "." in str(value) else str(value)
    if "." in text:
        return rf"(?<![\d.]){re.escape(text)}0*(?![\d.])"
    return rf"(?<![\d.]){re.escape(text)}(?:\.0+)?(?![\d.])"


def date_pattern(value):
    day = datetime.fromisoformat(value)
    return "(?:" + "|".join(re.escape(item) for item in (
        value, day.strftime("%B %d").replace(" 0", " "), day.strftime("%b %d").replace(" 0", " "),
    )) + ")"


def response_checks(result):
    case_id = result["test_id"]
    reference = result.get("expected_values", {})
    if case_id == "ev_charging_1":
        date = result["expected_args"]["get_electricity_prices"]["date"]
        usable = {
            review["id"] for review in result["tool_evaluation"]["call_reviews"]
            if review["usable"] and review["name"] == "get_weather_forecast"
        }
        peaks = []
        for call in result["tool_calls"]:
            if call["id"] not in usable:
                continue
            forecast = call["result"].get("daily_solar_peaks", [])
            if isinstance(forecast, list):
                peaks.extend(row for row in forecast if (
                    isinstance(row, dict) and row.get("date") == date
                    and type(row.get("hour")) is int and isinstance(row.get("solar_irradiance"), (int, float))
                ))
        peak_pattern = r"(?!)"
        if peaks:
            peak = peaks[0]
            hour = peak["hour"]
            hour_pattern = rf"(?:{hour:02d}:00|{(hour - 1) % 12 + 1}(?::00)?\s*{'AM' if hour < 12 else 'PM'})"
            irradiance = number_pattern(peak["solar_irradiance"]) + r"\s*W/m(?:²|2)"
            peak_pattern = rf"(?:{hour_pattern}[^\n]{{0,80}}{irradiance}|{irradiance}[^\n]{{0,80}}{hour_pattern})"
        return {
            "accuracy": [check("Correct date", date_pattern(date)), check("Forecast peak and hour", peak_pattern),
                         check("Off-peak rate and rate difference", r"(?=.*\b0\.12\b)(?=.*\b0\.16\b)")],
            "relevance": [check("Addresses EV charging", r"charg")],
            "completeness": [check("Early off-peak window", r"00:00\s*(?:[-–—]|to)\s*06:00"),
                             check("Late off-peak window", r"22:00\s*(?:[-–—]|to)\s*24:00"),
                             check("Labels simulated data", r"simulat")],
            "usefulness": [check("Conditions solar charging on surplus", r"surplus solar|solar surplus")],
        }
    if case_id == "dishwasher_shift":
        return {
            "accuracy": [check("Savings of USD 0.19 per cycle", r"\b0\.19(?:2)?\b"),
                         check("Same 1.2 kWh load", r"\b1\.2\s*kWh")],
            "relevance": [check("Addresses the dishwasher or cycle", r"dishwasher|cycle"),
                          check("Keeps the answer focused on tariffs", r"\bweather\b|irradiance", False)],
            "completeness": [check("Compares both start hours", r"(?=.*(?:19:00|7\s*PM))(?=.*(?:22:00|10\s*PM))")],
            "usefulness": [check("States savings per cycle", r"per cycle|each cycle|savings calculation")],
        }
    if case_id == "heatwave_hvac":
        return {
            "accuracy": [check("Avoids guarantees about comfort", r"without sacrificing comfort|guarantee", False)],
            "relevance": [check("Addresses heat-wave cooling", r"cooling|air conditioning|heat wave")],
            "completeness": [check("Cites a relevant guide", r"tip_(?:hvac_efficiency|seasonal_energy)\.txt")],
            "usefulness": [check("Offers an actionable measure", r"shade|curtain|filter|vent|fan"),
                           check("Addresses safe or comfortable conditions", r"safe|comfortable|comfort")],
        }
    if case_id == "historical_ev":
        usage = reference["query_energy_usage"]
        return {
            "accuracy": [check("EV total matches SQLite", number_pattern(usage["total_consumption_kwh"]) + r"\s*kWh"),
                         check("Record count matches SQLite", number_pattern(usage["total_records"]) + r"\s+(?:matching\s+)?records")],
            "relevance": [check("Addresses EV consumption", r"\bEV\b|electric vehicle")],
            "completeness": [check("Includes both requested output fields", r"(?=.*kWh)(?=.*records)")],
            "usefulness": [check("Reports a concise quantity with units", r"\btotal\b|used|consum")],
        }
    if case_id == "daily_solar":
        return {
            "accuracy": [check(f"Daily total for {day}", rf"{date_pattern(day)}[^\n]{{0,60}}{number_pattern(value)}\s*kWh")
                         for day, _, value in daily_solar_rows],
            "relevance": [check("Addresses solar production", r"solar|generat")],
            "completeness": [check("Identifies the highest date or tied dates",
                                   rf"(?:highest|most|top)[^\n]{{0,100}}{date_pattern(solar_leaders[0])}")
                             ] + [check(f"Includes {day}", date_pattern(day)) for day in solar_leaders[1:]],
            "usefulness": [check("Presents daily values in kWh", r"\bkWh\b")],
        }
    if case_id == "recent_summary":
        values = reference["get_recent_energy_summary"]
        hours = result["expected_args"]["get_recent_energy_summary"]["hours"]
        return {
            "accuracy": [check("Recent consumption matches SQLite", number_pattern(values["usage.total_consumption_kwh"]) + r"\s*kWh"),
                         check("Cost matches SQLite", number_pattern(values["usage.total_cost_usd"])),
                         check("Solar generation matches SQLite", number_pattern(values["generation.total_generation_kwh"]) + r"\s*kWh"),
                         check("Avoids contradicting available data", r"no (?:solar |generation |consumption )?(?:data|records)", False)],
            "relevance": [check("Uses the requested rolling window", number_pattern(hours) + r"\s*hours")],
            "completeness": [check("Labels consumption, cost, and generation", r"(?=.*consum)(?=.*cost)(?=.*generat)")],
            "usefulness": [check("Shows the measured window and kWh", r"(?=.*hours)(?=.*kWh)")],
        }
    if case_id == "hvac_savings":
        return {
            "accuracy": [check("Energy savings of 2 kWh", r"\b2(?:\.0+)?\s*kWh"),
                         check("Cost savings of USD 0.40", r"\b0\.40\b|\b0\.4\b"),
                         check("Reduction of 20 percent", r"\b20(?:\.0+)?\s*(?:%|percent)")],
            "relevance": [check("Avoids unrelated default-rate estimates", r"\b0\.24\b", False)],
            "completeness": [check("Gives the daily period", r"per day|daily|/day")],
            "usefulness": [check("Uses the given USD 0.20 rate", r"\b0\.2(?:0)?\b")],
        }
    if case_id == "home_automation":
        return {
            "accuracy": [check("Avoids claiming fixed typical off-peak hours", r"typically.{0,40}(?:AM|PM)", False)],
            "relevance": [check("Addresses occupancy, plugs, or scheduling", r"occupancy|smart plug|schedul")],
            "completeness": [check("Cites the automation guide", r"tip_home_automation\.txt")],
            "usefulness": [check("Offers a concrete rule or measurement", r"turn off|schedule|monitor|standby")],
        }
    if case_id == "battery_runtime":
        return {
            "accuracy": [check("Estimates about four hours", r"\b(?:4(?:\.0+)?|four)\s+hours?"),
                         check("Uses the stated available energy without requiring full charge", r"fully charged", False)],
            "relevance": [check("Discusses the battery or load", r"\bbattery\b|\bload\b")],
            "completeness": [check("Cites the retrieved battery guide", r"tip_battery_storage\.txt"),
                             check("Uses energy available above reserve", r"reserve|usable capacity")],
            "usefulness": [check("States the steady 1 kW load", r"\b1\s*kW\b"), check("Explains losses or efficiency", r"loss|efficien")],
        }
    if case_id == "missing_history":
        return {
            "accuracy": [check("Identifies missing consumption data", r"no (?:recorded |consumption |usage )?(?:data|records)|no recorded (?:electricity )?(?:usage|consumption)|missing data|unavailable"),
                         check("Avoids reporting measured zero", r"\b0\s*kWh", False)],
            "relevance": [check("Focuses on consumption", r"usage|consumption|electricity"),
                          check("Avoids unrelated solar claims", r"\bsolar\b", False)],
            "completeness": [check("Includes the requested date", date_pattern(result["expected_args"]["query_energy_usage"]["start_date"]))],
            "usefulness": [check("Makes the data gap clear", r"no |missing|unavailable")],
        }
    if case_id == "energy_upgrade_roi":
        return {
            "accuracy": [
                check("Annual bill savings of USD 146", r"\b146(?:\.0+)?\b"),
                check("Simple payback of about 0.68 years or 8.2 months", r"\b0\.68(?:\s*years?)?|\b8\.2\s*months?"),
                check("Two-year ROI of 192 percent", r"\b192(?:\.0+)?\s*(?:%|percent)"),
                check("Annual avoided emissions of 292 kg CO2e", r"\b292(?:\.0+)?\s*kg\s*CO(?:2|₂)e"),
            ],
            "relevance": [check("Addresses the upgrade investment", r"\bROI\b|payback|invest")],
            "completeness": [check("Labels all requested outcomes", r"(?=.*payback)(?=.*ROI)(?=.*emissions)(?=.*savings)")],
            "usefulness": [check("Makes the daily-to-annual assumption explicit", r"365 days|(?:daily|each day|per day).{0,80}(?:annual|year)"),
                           check("Uses a two-year horizon", r"(?:two|2)[ -]years?")],
        }
    raise ValueError(f"No response checks are defined for {case_id}.")

In [13]:
import re

RESPONSE_CRITERIA = ("accuracy", "relevance", "completeness", "usefulness")

def evaluate_response(question, final_response, expected_response):
    """Score each answer against the observable checks for its question."""
    if not question.strip() or not final_response.strip():
        return {
            "scores": dict.fromkeys(RESPONSE_CRITERIA, 0.0),
            "feedback": "No answer was available for evaluation.",
            "improvement": "Inspect the execution error and rerun the scenario.",
        }

    answer = re.sub(r"(?<=\d),(?=\d{3}\b)", "", final_response)
    scores = {}
    feedback = []
    for criterion in RESPONSE_CRITERIA:
        checks = expected_response["checks"][criterion]
        passed = []
        missed = []
        for label, pattern, expected in checks:
            matched = re.search(pattern, answer, flags=re.IGNORECASE | re.DOTALL) is not None
            (passed if matched == expected else missed).append(label)
        scores[criterion] = 1.0 if passed and not missed else 0.5 if passed else 0.0
        feedback.append(f"{criterion.capitalize()} {scores[criterion]:.2f}")
        if passed:
            feedback.append("Met " + ", ".join(passed))
        if missed:
            feedback.append("Needs " + ", ".join(missed))
    return {
        "scores": scores,
        "feedback": ". ".join(feedback) + ".",
        "improvement": ("Check the unmet criteria against this reference. " + expected_response["description"]
                        if any(score < 1.0 for score in scores.values()) else ""),
    }

### Evaluate tool use from the message trace

| Metric | Calculation |
| --- | --- |
| Tool appropriateness | Appropriate calls divided by all attempted calls. A call must use an expected or permitted optional tool with relevant arguments. Repeating the same call after it provided usable evidence reduces this score. |
| Tool completeness | Required tools that provided usable evidence divided by all required tools. The result must succeed and meet any reference checks for values or sources. |

The call ID connects each request to its own result. A well-chosen call that fails still shows appropriate selection. It contributes to completeness only after a successful attempt. A range query can satisfy the solar scenario in one call, or the agent can assemble the result from separate valid queries for every requested day. When the scenario requires no tools, an empty trace earns full credit.

The [controlled tests](tests/test_tool_evaluation.py) use synthetic message traces to verify the scores for missing, extra, failed, redundant, and wrong-argument calls. They also exercise retries, reference values, and source citations without a provider call.

In [14]:
def matches_reference(actual, expected):
    if isinstance(expected, range):
        return isinstance(actual, int) and actual in expected
    if (
        isinstance(actual, (int, float)) and isinstance(expected, (int, float))
        and not isinstance(actual, bool) and not isinstance(expected, bool)
    ):
        return isclose(actual, expected, rel_tol=0, abs_tol=0.005)
    return type(actual) is type(expected) and actual == expected


def tool_result_issues(call, reference, expected_sources):
    issues = []
    for field, expected in reference.items():
        actual = call["result"]
        for part in field.split("."):
            if isinstance(actual, dict):
                actual = actual.get(part)
            elif isinstance(actual, list) and part.isdecimal() and int(part) < len(actual):
                actual = actual[int(part)]
            else:
                actual = None
                break
        if not matches_reference(actual, expected):
            issues.append(f"{field}: expected {expected!r}, received {actual!r}")

    if call["name"] == "search_energy_tips" and expected_sources:
        tips = call["result"].get("tips", [])
        sources = {
            Path(tip["source"]).name for tip in tips
            if isinstance(tip, dict) and isinstance(tip.get("source"), str)
        } if isinstance(tips, list) else set()
        if sources.isdisjoint(expected_sources):
            issues.append(
                f"Expected a guide from {', '.join(sorted(expected_sources))}; "
                f"retrieved {', '.join(sorted(sources)) or 'none'}"
            )
    return issues


def evaluate_tool_usage(messages_list, expected_tools, expected_args=None, expected_values=None,
                        optional_tools=(), expected_sources=()):
    calls = extract_tool_calls(messages_list)
    required = set(expected_tools)
    allowed = required | set(optional_tools)
    expected_args = expected_args or {}
    expected_values = expected_values or {}
    observed = {call["name"] for call in calls}
    covered = set()
    usable_signatures = set()
    call_reviews = []
    missing = required - observed
    issues = [f"Missing required tool: {name}" for name in sorted(missing)]

    solar_tool = "query_solar_generation"
    solar_args = expected_args.get(solar_tool, {})
    solar_dates = set()
    daily_results = {}
    if solar_tool in required and {"start_date", "end_date"} <= solar_args.keys():
        start = datetime.fromisoformat(solar_args["start_date"]).date()
        end = datetime.fromisoformat(solar_args["end_date"]).date()
        solar_dates = {(start + timedelta(days=offset)).isoformat()
                       for offset in range((end - start).days + 1)}

    argument_defaults = {"get_weather_forecast": {"days": 3}, "query_energy_usage": {"device_type": None}}
    for call in calls:
        name, args = call["name"], call["args"]
        resolved_args = {**argument_defaults.get(name, {}), **args}
        required_args = expected_args.get(name, {})
        signature = (name, json.dumps(resolved_args, sort_keys=True))
        mismatches = {
            key: expected for key, expected in required_args.items()
            if not matches_reference(resolved_args.get(key), expected)
        }
        day = args.get("start_date")
        daily_query = (
            name == solar_tool and bool(mismatches)
            and mismatches.keys() <= {"start_date", "end_date"}
            and isinstance(day, str) and day in solar_dates and args.get("end_date") == day
        )

        selection_issue = None
        if name not in allowed:
            selection_issue = "Unexpected tool for this scenario"
        elif mismatches and not daily_query:
            selection_issue = f"Wrong arguments: expected {mismatches!r}, received {args!r}"
        elif signature in usable_signatures or (daily_query and (name in covered or day in daily_results)):
            selection_issue = "Redundant call after a usable result"

        result_issues = []
        if call["error"]:
            result_issues.append(call["error"])
        elif selection_issue is None:
            reference = expected_values.get(name, {})
            if daily_query:
                reference = {
                    field: expected for field, expected in reference.items()
                    if field not in {"total_records", "daily_generation"}
                }
                daily_reference = expected_values.get(name, {}).get("daily_generation")
                if daily_reference is not None:
                    rows = [row for row in daily_reference if row["date"] == day]
                    reference["daily_generation"] = rows
                    reference["total_records"] = sum(row["total_records"] for row in rows)
                record_count = call["result"].get("total_records")
                if type(record_count) is not int or record_count < 0:
                    result_issues.append("Daily solar result needs a nonnegative record count")
            result_issues.extend(tool_result_issues(call, reference, expected_sources))
            if not result_issues:
                usable_signatures.add(signature)
                if daily_query:
                    daily_results[day] = record_count
                elif name in required:
                    covered.add(name)

        reasons = ([selection_issue] if selection_issue else []) + result_issues
        issues.extend(f"{name} ({call['id']}): {reason}" for reason in reasons)
        call_reviews.append({
            "id": call["id"], "name": name, "args": args,
            "appropriate": selection_issue is None,
            "usable": not reasons,
            "feedback": reasons or (["Usable evidence for one day"] if daily_query else ["Usable result"]),
        })

    if daily_results and solar_tool not in covered:
        missing_days = solar_dates - daily_results.keys()
        expected_count = expected_values.get(solar_tool, {}).get("total_records")
        actual_count = sum(daily_results.values())
        if missing_days:
            issues.append(f"{solar_tool}: missing usable evidence for {', '.join(sorted(missing_days))}")
        elif expected_count is not None and actual_count != expected_count:
            issues.append(f"{solar_tool}: daily calls returned {actual_count} records, expected {expected_count}")
        else:
            covered.add(solar_tool)

    appropriate = sum(review["appropriate"] for review in call_reviews)
    unusable = (required & observed) - covered
    feedback = [
        f"Appropriate calls: {appropriate}/{len(calls)}",
        f"Required tools with usable evidence: {', '.join(sorted(covered)) or 'none'} ({len(covered)}/{len(required)})",
    ]
    if missing:
        feedback.append(f"Required tools not called: {', '.join(sorted(missing))}")
    if unusable:
        feedback.append(f"Required tools called without usable evidence: {', '.join(sorted(unusable))}")
    feedback.extend(issues)
    return {
        "appropriateness": appropriate / len(calls) if calls else float(not required),
        "completeness": len(covered) / len(required) if required else 1.0,
        "covered_tools": sorted(covered),
        "missing_tools": sorted(missing),
        "unusable_tools": sorted(unusable),
        "call_reviews": call_reviews,
        "issues": issues,
        "feedback": feedback,
    }

### Read the report

The report shows four response scores and two tool scores on the same 0–1 scale. Read the findings below the table to see which checks passed, which calls were missing or unusable, and which expectation to address next. The call-level details are also stored in each entry of `test_results`. The controlled unit tests and the live scenario run measure different things: the tests verify the evaluator's scoring rules, while the scenarios show what the model did in this run.

In [15]:
def generate_evaluation_report(results):
    lines = [
        "## Energy Advisor evaluation",
        "Response scores are **accuracy / relevance / completeness / usefulness**. All six scores range from 0 to 1.",
        "Tool appropriateness scores the calls selected. Tool completeness scores the required evidence obtained.",
        "",
        "| Scenario | Tools called | Tool appropriateness | Tool completeness | Tool checks | Response scores |",
        "| --- | --- | ---: | ---: | --- | --- |",
    ]
    details = []
    scores = []
    clean_tools = 0
    completeness = []
    appropriateness = []
    for result in results:
        messages = result["response"]["messages"] if result["response"] else []
        values = {} if result.get("reference_window_changed") else result.get("expected_values")
        tool_review = evaluate_tool_usage(
            messages, result["expected_tools"], result.get("expected_args"), values,
            result.get("optional_tools", ()), result.get("expected_sources", ()),
        )
        result["tool_evaluation"] = tool_review
        expectation = {"description": result["expected_response"], "checks": response_checks(result)}
        response_review = evaluate_response(result["question"], result["final_response"], expectation)
        appropriateness.append(tool_review["appropriateness"])
        completeness.append(tool_review["completeness"])
        result["response_evaluation"] = response_review
        tools_called = ", ".join(f"`{call['name']}`" for call in result["tool_calls"]) or "—"
        issues = tool_review["issues"]
        checked = "Pass" if not issues else f"{len(issues)} finding(s)"
        if result.get("reference_window_changed"):
            checked += " (SQL window moved; tool value checks skipped)"
        clean_tools += not issues
        score = response_review["scores"]
        scores.append(score)
        score_text = " / ".join(f"{score[key]:.2f}" for key in RESPONSE_CRITERIA)
        lines.append(
            f"| {result['label']} | {tools_called} | {tool_review['appropriateness']:.2f} | "
            f"{tool_review['completeness']:.2f} | {checked} | {score_text} |"
        )
        details.extend([
            f"- **{result['label']}**",
            f"  - **Tools.** {' · '.join(tool_review['feedback'])}.",
            f"  - **Response.** {response_review['feedback']}",
        ])
        if response_review["improvement"]:
            details.append(f"  - **Suggested improvement.** {response_review['improvement']}")

    if scores:
        full_tools = sum(value == 1.0 for value in completeness)
        lines.extend([
            "",
            f"All required tools provided usable evidence in **{full_tools}/{len(results)}** scenarios. "
            f"Tool checks passed for **{clean_tools}/{len(results)}** scenarios.",
            f"Mean tool appropriateness **{sum(appropriateness) / len(results):.2f}**. "
            f"Mean tool completeness **{sum(completeness) / len(results):.2f}**.",
        ])
        fully_met = sum(all(value == 1.0 for value in score.values()) for score in scores)
        lines.append(f"**{fully_met}/{len(scores)}** responses met all four criteria.")
        averages = ", ".join(
            f"{key} {sum(item[key] for item in scores) / len(scores):.2f}" for key in RESPONSE_CRITERIA
        )
        lines.append(f"Scored responses **{len(scores)}/{len(results)}**. Average scores: {averages}.")
    lines.extend(["", "### Findings and improvements", "", *details])
    return "\n".join(lines)


def display_evaluation_report(report):
    display(Markdown(report))

In [16]:
display_evaluation_report(generate_evaluation_report(test_results))

## Energy Advisor evaluation
Response scores are **accuracy / relevance / completeness / usefulness**. All six scores range from 0 to 1.
Tool appropriateness scores the calls selected. Tool completeness scores the required evidence obtained.

| Scenario | Tools called | Tool appropriateness | Tool completeness | Tool checks | Response scores |
| --- | --- | ---: | ---: | --- | --- |
| EV charging with solar and tariffs | `get_weather_forecast`, `get_electricity_prices` | 1.00 | 1.00 | Pass | 1.00 / 1.00 / 1.00 / 1.00 |
| Dishwasher time-of-use savings | `get_electricity_prices`, `get_weather_forecast` | 0.50 | 1.00 | 1 finding(s) | 1.00 / 1.00 / 1.00 / 1.00 |
| Cooling during a heat wave | `search_energy_tips` | 1.00 | 1.00 | Pass | 1.00 / 1.00 / 1.00 / 1.00 |
| Historical EV consumption | `query_energy_usage` | 1.00 | 1.00 | Pass | 1.00 / 1.00 / 1.00 / 1.00 |
| Daily solar generation | `query_solar_generation` | 1.00 | 1.00 | Pass | 1.00 / 1.00 / 1.00 / 1.00 |
| Recent household summary | `get_recent_energy_summary` | 1.00 | 1.00 | Pass | 1.00 / 1.00 / 1.00 / 1.00 |
| Savings from reducing HVAC usage | `calculate_energy_savings` | 1.00 | 1.00 | Pass | 1.00 / 1.00 / 1.00 / 0.00 |
| Home automation | `search_energy_tips` | 1.00 | 1.00 | Pass | 1.00 / 1.00 / 1.00 / 1.00 |
| Home battery runtime | `search_energy_tips` | 1.00 | 1.00 | Pass | 1.00 / 1.00 / 1.00 / 1.00 |
| Period without usage records | `query_energy_usage` | 1.00 | 1.00 | Pass | 1.00 / 1.00 / 1.00 / 1.00 |
| Upgrade ROI and avoided emissions | `calculate_energy_savings` | 1.00 | 1.00 | Pass | 1.00 / 1.00 / 1.00 / 0.50 |

All required tools provided usable evidence in **11/11** scenarios. Tool checks passed for **10/11** scenarios.
Mean tool appropriateness **0.95**. Mean tool completeness **1.00**.
**9/11** responses met all four criteria.
Scored responses **11/11**. Average scores: accuracy 1.00, relevance 1.00, completeness 1.00, usefulness 0.86.

### Findings and improvements

- **EV charging with solar and tariffs**
  - **Tools.** Appropriate calls: 2/2 · Required tools with usable evidence: get_electricity_prices, get_weather_forecast (2/2).
  - **Response.** Accuracy 1.00. Met Correct date, Forecast peak and hour, Off-peak rate and rate difference. Relevance 1.00. Met Addresses EV charging. Completeness 1.00. Met Early off-peak window, Late off-peak window, Labels simulated data. Usefulness 1.00. Met Conditions solar charging on surplus.
- **Dishwasher time-of-use savings**
  - **Tools.** Appropriate calls: 1/2 · Required tools with usable evidence: get_electricity_prices (1/1) · get_weather_forecast (call_k9uLMD8JSpez62ifd8aImOu4): Unexpected tool for this scenario.
  - **Response.** Accuracy 1.00. Met Savings of USD 0.19 per cycle, Same 1.2 kWh load. Relevance 1.00. Met Addresses the dishwasher or cycle, Keeps the answer focused on tariffs. Completeness 1.00. Met Compares both start hours. Usefulness 1.00. Met States savings per cycle.
- **Cooling during a heat wave**
  - **Tools.** Appropriate calls: 1/1 · Required tools with usable evidence: search_energy_tips (1/1).
  - **Response.** Accuracy 1.00. Met Avoids guarantees about comfort. Relevance 1.00. Met Addresses heat-wave cooling. Completeness 1.00. Met Cites a relevant guide. Usefulness 1.00. Met Offers an actionable measure, Addresses safe or comfortable conditions.
- **Historical EV consumption**
  - **Tools.** Appropriate calls: 1/1 · Required tools with usable evidence: query_energy_usage (1/1).
  - **Response.** Accuracy 1.00. Met EV total matches SQLite, Record count matches SQLite. Relevance 1.00. Met Addresses EV consumption. Completeness 1.00. Met Includes both requested output fields. Usefulness 1.00. Met Reports a concise quantity with units.
- **Daily solar generation**
  - **Tools.** Appropriate calls: 1/1 · Required tools with usable evidence: query_solar_generation (1/1).
  - **Response.** Accuracy 1.00. Met Daily total for 2026-11-15, Daily total for 2026-11-16, Daily total for 2026-11-17. Relevance 1.00. Met Addresses solar production. Completeness 1.00. Met Identifies the highest date or tied dates. Usefulness 1.00. Met Presents daily values in kWh.
- **Recent household summary**
  - **Tools.** Appropriate calls: 1/1 · Required tools with usable evidence: get_recent_energy_summary (1/1).
  - **Response.** Accuracy 1.00. Met Recent consumption matches SQLite, Cost matches SQLite, Solar generation matches SQLite, Avoids contradicting available data. Relevance 1.00. Met Uses the requested rolling window. Completeness 1.00. Met Labels consumption, cost, and generation. Usefulness 1.00. Met Shows the measured window and kWh.
- **Savings from reducing HVAC usage**
  - **Tools.** Appropriate calls: 1/1 · Required tools with usable evidence: calculate_energy_savings (1/1).
  - **Response.** Accuracy 1.00. Met Energy savings of 2 kWh, Cost savings of USD 0.40, Reduction of 20 percent. Relevance 1.00. Met Avoids unrelated default-rate estimates. Completeness 1.00. Met Gives the daily period. Usefulness 0.00. Needs Uses the given USD 0.20 rate.
  - **Suggested improvement.** Check the unmet criteria against this reference. 2 kWh/day, USD 0.40/day, and 20% less energy.
- **Home automation**
  - **Tools.** Appropriate calls: 1/1 · Required tools with usable evidence: search_energy_tips (1/1).
  - **Response.** Accuracy 1.00. Met Avoids claiming fixed typical off-peak hours. Relevance 1.00. Met Addresses occupancy, plugs, or scheduling. Completeness 1.00. Met Cites the automation guide. Usefulness 1.00. Met Offers a concrete rule or measurement.
- **Home battery runtime**
  - **Tools.** Appropriate calls: 1/1 · Required tools with usable evidence: search_energy_tips (1/1).
  - **Response.** Accuracy 1.00. Met Estimates about four hours, Uses the stated available energy without requiring full charge. Relevance 1.00. Met Discusses the battery or load. Completeness 1.00. Met Cites the retrieved battery guide, Uses energy available above reserve. Usefulness 1.00. Met States the steady 1 kW load, Explains losses or efficiency.
- **Period without usage records**
  - **Tools.** Appropriate calls: 1/1 · Required tools with usable evidence: query_energy_usage (1/1).
  - **Response.** Accuracy 1.00. Met Identifies missing consumption data, Avoids reporting measured zero. Relevance 1.00. Met Focuses on consumption, Avoids unrelated solar claims. Completeness 1.00. Met Includes the requested date. Usefulness 1.00. Met Makes the data gap clear.
- **Upgrade ROI and avoided emissions**
  - **Tools.** Appropriate calls: 1/1 · Required tools with usable evidence: calculate_energy_savings (1/1).
  - **Response.** Accuracy 1.00. Met Annual bill savings of USD 146, Simple payback of about 0.68 years or 8.2 months, Two-year ROI of 192 percent, Annual avoided emissions of 292 kg CO2e. Relevance 1.00. Met Addresses the upgrade investment. Completeness 1.00. Met Labels all requested outcomes. Usefulness 0.50. Met Uses a two-year horizon. Needs Makes the daily-to-annual assumption explicit.
  - **Suggested improvement.** Check the unmet criteria against this reference. USD 146 annual savings, about 0.68 years simple payback, 192% ROI over two years, and 292 kg CO2e avoided annually. Use the given factor and assume 365 days of savings.

## 5. Targeted regression retest

Retested on 2026-11-19. These two targeted checks were run after the historical query and recent-summary prompt corrections and before the full rerun.

| Scenario | Tool called | Verified result |
| --- | --- | --- |
| Historical EV consumption | `query_energy_usage` with `start_date=2026-11-15`, `end_date=2026-11-16`, and `device_type=EV` | 48 records and 414.63 kWh. Both matched the independent SQLite reference. |
| Recent household summary | `get_recent_energy_summary` with `hours=66` | 324.05 kWh consumed, USD 36.52 in cost, and 17.40 kWh generated. The tool and final answer agreed with the SQLite reference for the requested window. |

Each scenario used one tool call. The recent window is selected at runtime to include the most recent sample data, so it was 66 hours on this retest.

### Historical EV consumption response

From November 15 to November 16, 2026, the electric vehicle (EV) used a total of **414.63 kWh** across **48 records**.

### Recent household summary response

Over the last 66 hours, your household's totals are as follows:

- **Total Electricity Consumption:** 324.05 kWh
- **Total Cost:** $36.52
- **Total Solar Generation:** 17.4 kWh

If you have any further questions or need assistance with planning your energy use, feel free to ask!

## 6. Comparison with the initial evaluation

The initial run and the full rerun on 2026-11-19 used the same ten scenarios and review criteria. Historical response averages were divided by two to express them on the 0–1 scale. The historical tool counts retain the earlier evaluator's definitions. The recent-summary window was 65 hours initially and 68 hours on the full rerun. The window is selected at runtime to reach the sample data, and each result was checked against its own SQLite reference.

| Metric | Initial run | Full rerun |
| --- | ---: | ---: |
| Cases with all required tools returning successfully | 9/10 | 9/10 |
| Cases passing all tool checks | 5/10 | 5/10 |
| Responses meeting all four criteria | 3/10 | 3/10 |
| Average accuracy | 0.70 | 0.70 |
| Average relevance | 0.85 | 0.85 |
| Average completeness | 0.75 | 0.85 |
| Average usefulness | 0.75 | 0.75 |

| Scenario | Initial run | Full rerun |
| --- | --- | --- |
| EV charging | Correct tariff and forecast peak. | Correct tariffs, wrong irradiance peak. The forecast showed 534 W/m² at 11:00. |
| Dishwasher | Correct savings with an unnecessary weather call and solar discussion. | Correct savings and focused answer. The unnecessary weather call persisted. |
| Cooling during a heat wave | Practical tips with an external reference, no guide filename. | Cited the retrieved files. The blanket 4°F comfort claim remained. |
| Historical EV consumption | 49 records and 422.94 kWh after including the following midnight. | 48 records and 414.63 kWh, matching SQLite. |
| Daily solar generation | Correct daily totals and highest day from three solar queries. | Used the 18.01 kWh average for every day and called the unrelated consumption tool. |
| Recent household summary | Correct totals followed by a contradictory history lookup for 2023. | One summary call for 68 hours. All three totals matched SQLite. |
| HVAC savings | Correct answer plus an extra calculation with the default tariff. | The same redundant default-rate calculation persisted. |
| Home automation | Useful routines, missing source filename, and an unverified EV rate window. | The same source and tariff issues persisted. |
| Battery runtime | Correct four-hour estimate, missing RAG lookup and citation. | The same retrieval and citation gap persisted. |
| Period without usage records | Correct missing-data answer with one consumption lookup. | Correct missing-data answer with an unnecessary solar lookup. |

### Remaining issues

- Return and use explicit daily generation totals and the target date peak irradiance so the model has values it can verify directly.
- Require retrieval when the user requests a guide citation and cite the retrieved filename.
- Avoid weather or unrelated history lookups for single-purpose questions, and honor the rate supplied by the user.
- Keep heat-wave guidance conditional on safe indoor conditions and avoid presenting unverified tariff windows as typical advice for the household.

These results describe one full rerun. They show the observed changes without attributing every difference to the two corrections.

## 7. Solar regression retest

Retested on 2026-11-19. These two cases check how the agent uses the computed solar fields. The ten-case comparison above records the earlier full rerun.

| Scenario | Tools called | Verified answer |
| --- | --- | --- |
| EV charging | `get_weather_forecast`, `get_electricity_prices` | The peak on 2026-11-20 was 534 W/m² at 11:00. The answer matched both the daily peak field and the hourly forecast. |
| Daily generation | `query_solar_generation` | 2026-11-15: 18.64 kWh, 2026-11-16: 17.99 kWh, 2026-11-17: 17.40 kWh. Highest generation on 2026-11-15. The answer matched the data grouped by date in SQLite. |

The EV answer used the correct irradiance peak and rate values. It displayed the last included hour as the end of each off-peak interval. The simulated rates apply from 00:00 through 06:00 and from 22:00 through 24:00, with exclusive end times. This retest passed the solar checks and exposed that tariff presentation issue.

### EV charging response

Here’s the best plan for charging your electric vehicle tomorrow, November 20, 2026, to minimize costs and maximize solar power:

- **Scenario**: November 20, 2026 (Simulated data)
- **Solar**: The peak solar irradiance is expected at 11:00 AM with a value of 534 W/m². Charging your EV during this time could be beneficial if your home has surplus solar generation.
- **Grid**: The lowest electricity rates are during the off-peak hours:
  - 00:00 - 05:00: $0.12/kWh
  - 22:00 - 23:00: $0.12/kWh
- **Cost difference**: The peak rate from 06:00 to 21:00 is $0.28/kWh. The difference between the peak and off-peak rates is $0.16/kWh. If you know your EV charging load in kWh, you can calculate the savings by multiplying this difference by your load.

To maximize solar use, consider charging your EV around 11:00 AM when solar generation is at its peak, provided you have surplus solar energy available. Otherwise, charging during the off-peak hours (especially overnight) will save you money.

### Daily solar generation response

Here are the solar energy generation totals for each day between November 15 and November 17, 2026:

- **November 15, 2026**: 18.64 kWh
- **November 16, 2026**: 17.99 kWh
- **November 17, 2026**: 17.40 kWh

The day with the highest solar generation was **November 15, 2026**.

## 8. Tariff window retest

Retested the EV charging scenario on 2026-11-19 for 2026-11-20. The tool calls were `get_weather_forecast` followed by `get_electricity_prices`.

The tariff tool returned **00:00–06:00 and 22:00–24:00** as the lowest-rate windows at **USD 0.12/kWh**. The agent reported both intervals with their exclusive end times. The two windows cover hours 0 through 5 and 22 through 23.

### EV charging response

Here’s the best plan for charging your electric car tomorrow, November 20, 2026, to minimize costs and maximize solar power:

- **Scenario**: November 20, 2026 (Simulated)
- **Solar**: The peak solar irradiance is expected at **11:00** with **534 W/m²**. Charging during this time is beneficial if your home has surplus solar generation.
- **Grid**: The lowest electricity rate is **$0.12/kWh** during the following windows:
  - **00:00–06:00**
  - **22:00–24:00**
- **Cost difference**: The peak rate is **$0.28/kWh**, resulting in a difference of **$0.16/kWh**. If you charge during peak hours, you would incur an additional cost of **$0.16** for every kWh used compared to charging during off-peak hours.

To maximize savings, consider charging your electric vehicle during the off-peak hours (before 6 AM or after 10 PM) and align your charging with the solar peak around 11 AM if you have surplus solar generation.